# Vision Arena with Eden AI

Drop an image, ask a question, watch four vision LLMs stream their answers side-by-side. Vote on the winner or let an LLM judge pick.

Same API, same key, same response format — only the `model` string and the `content` block shape change. This is the [LLM Arena](llm_arena.ipynb), extended to multimodal.

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var or in `.env`).

In [ ]:
%pip install --quiet aiohttp ipywidgets nest_asyncio python-dotenv requests

## 1. Configuration

Each entry is one vision-capable model. Add or remove rows freely — the grid auto-resizes. Every model below has `image` in its `input_modalities` per the Eden AI catalog.

In [ ]:
import base64
import json
import os

from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(override=True)

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY")
if not EDENAI_API_KEY:
    raise RuntimeError("Set EDENAI_API_KEY (env var or .env file). Get one at https://app.edenai.run")
EDENAI_URL = "https://api.edenai.run/v3/llm/chat/completions"

MODELS = [
    {"label": "Claude",  "model": "anthropic/claude-sonnet-4-5"},
    {"label": "GPT-4o",  "model": "openai/gpt-4o"},
    {"label": "Gemini",  "model": "google/gemini-2.5-flash"},
    {"label": "Pixtral", "model": "mistral/pixtral-large-latest"},
]

JUDGE_MODEL = "anthropic/claude-sonnet-4-5"

DEFAULT_IMAGE_URL = "https://picsum.photos/seed/cookbook/640/480.jpg"


def _is_sandbox(jwt: str) -> bool:
    try:
        payload_b64 = jwt.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64)).get("type") == "sandbox_api_token"
    except Exception:
        return False


if _is_sandbox(EDENAI_API_KEY):
    display(HTML(
        '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 14px;'
        'border-radius:4px;font-family:sans-serif;font-size:13px;margin:6px 0;">'
        '<b>⚠ Sandbox key detected.</b> Vision inputs still hit the real providers '
        '(unlike text-only sandbox calls), so the arena will show real differences. '
        'Switch to a production key for higher rate limits.</div>'
    ))

## 2. Streaming caller (multimodal)

OpenAI-compatible chat completions accept a list of content blocks per message. To send an image we use:

```python
{"role": "user", "content": [
    {"type": "text",      "text": "What's in this image?"},
    {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64,..."}},
]}
```

The image can be a base64 data URI or a publicly reachable HTTPS URL. Everything else (SSE streaming, latency badges) is identical to the text arena.

In [ ]:
import time

import aiohttp


def _header_html(label, model, status="idle", status_color="#6c757d", latency=None):
    lat = f" · {latency:.2f}s" if latency is not None else ""
    return (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px 2px;">'
        f'  <div>'
        f'    <span style="font-weight:600;font-size:14px;">{label}</span>'
        f'    <span style="color:#888;font-size:11px;margin-left:6px;">{model}</span>'
        '  </div>'
        '  <div>'
        f'    <span style="background:{status_color};color:white;padding:3px 10px;'
        f'border-radius:10px;font-size:11px;font-weight:600;">{status}{lat}</span>'
        '  </div>'
        '</div>'
    )


def _build_messages(prompt_text, image_url):
    return [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt_text},
            {"type": "image_url", "image_url": {"url": image_url}},
        ],
    }]


async def stream_model(session, model_cfg, prompt_text, image_url, output_widget, header_widget):
    headers = {
        "Authorization": f"Bearer {EDENAI_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model_cfg["model"],
        "messages": _build_messages(prompt_text, image_url),
        "stream": True,
    }

    output_widget.clear_output()
    full_text = []
    first_token_at = None
    start = time.perf_counter()
    header_widget.value = _header_html(model_cfg["label"], model_cfg["model"], "streaming…", "#17a2b8")

    async with session.post(EDENAI_URL, headers=headers, json=payload) as resp:
        if resp.status != 200:
            header_widget.value = _header_html(model_cfg["label"], model_cfg["model"], "error ✗", "#dc3545")
            with output_widget:
                print(f"[error {resp.status}] {await resp.text()}")
            return {"label": model_cfg["label"], "text": "", "first_token": None, "total": None}

        async for raw in resp.content:
            line = raw.decode("utf-8").strip()
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            try:
                chunk = json.loads(data)
            except json.JSONDecodeError:
                continue
            token = chunk.get("choices", [{}])[0].get("delta", {}).get("content", "")
            if not token:
                continue
            if first_token_at is None:
                first_token_at = time.perf_counter() - start
            full_text.append(token)
            with output_widget:
                print(token, end="")
            header_widget.value = _header_html(
                model_cfg["label"], model_cfg["model"],
                "streaming…", "#17a2b8", time.perf_counter() - start,
            )

    total = time.perf_counter() - start
    header_widget.value = _header_html(model_cfg["label"], model_cfg["model"], "done ✓", "#28a745", total)
    return {
        "label": model_cfg["label"],
        "text": "".join(full_text),
        "first_token": first_token_at,
        "total": total,
    }

## 3. Image input + Arena UI

Two ways to supply an image:
- **Upload** a file from disk
- **Paste** a URL pointing to a publicly reachable image (PNG, JPEG, WEBP, GIF)

Whichever you set last wins. A default sample image is preloaded so you can press Fight right away.

In [ ]:
import requests

from ipywidgets import (
    Button, FileUpload, GridBox, HBox, HTML as HTMLWidget,
    Layout, Output, Text, Textarea, VBox,
)
from IPython.display import display

image_url_box = Text(
    value=DEFAULT_IMAGE_URL,
    placeholder="https://example.com/image.jpg",
    layout=Layout(width="100%"),
)
image_upload = FileUpload(accept="image/*", multiple=False, description="Upload image")
image_preview = HTMLWidget(
    value=(
        f'<img src="{DEFAULT_IMAGE_URL}" '
        'style="max-width:280px;max-height:200px;border:1px solid #ddd;border-radius:4px;" />'
    )
)

_current_image = {"data_uri": None, "source_url": DEFAULT_IMAGE_URL}


def _set_image_from_bytes(img_bytes, mime="image/jpeg"):
    b64 = base64.b64encode(img_bytes).decode()
    data_uri = f"data:{mime};base64,{b64}"
    _current_image["data_uri"] = data_uri
    _current_image["source_url"] = None
    image_preview.value = (
        f'<img src="{data_uri}" '
        'style="max-width:280px;max-height:200px;border:1px solid #ddd;border-radius:4px;" />'
    )


def _on_upload_change(change):
    files = change["new"]
    if not files:
        return
    f = files[0] if isinstance(files, (list, tuple)) else next(iter(files.values()))
    content = f["content"] if isinstance(f, dict) else f.content
    mime = (f["type"] if isinstance(f, dict) else f.type) or "image/jpeg"
    _set_image_from_bytes(bytes(content), mime)


def _on_url_change(change):
    new_url = (change["new"] or "").strip()
    if not new_url:
        return
    _current_image["data_uri"] = None
    _current_image["source_url"] = new_url
    image_preview.value = (
        f'<img src="{new_url}" '
        'style="max-width:280px;max-height:200px;border:1px solid #ddd;border-radius:4px;" />'
    )


image_upload.observe(_on_upload_change, names="value")
image_url_box.observe(_on_url_change, names="value")


def _resolve_image_url():
    """Return a value usable as the `image_url.url` field.
    Prefers an uploaded data URI; otherwise fetches the URL and inlines it as a data URI
    so providers that can't reach arbitrary hosts still work."""
    if _current_image["data_uri"]:
        return _current_image["data_uri"]
    url = _current_image["source_url"]
    if not url:
        return None
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    mime = r.headers.get("Content-Type", "image/jpeg").split(";")[0].strip()
    if not mime.startswith("image/"):
        mime = "image/jpeg"
    b64 = base64.b64encode(r.content).decode()
    return f"data:{mime};base64,{b64}"


prompt_box = Textarea(
    value="Describe this image in one sentence.",
    placeholder="Ask anything about the image…",
    layout=Layout(width="100%", height="60px"),
)
fight_btn = Button(description="⚔  Fight", button_style="primary")
clear_btn = Button(description="Clear panels")

panel_layout = Layout(border="1px solid #ddd", padding="8px", height="220px", overflow="auto")
panels = [Output(layout=panel_layout) for _ in MODELS]
headers = [HTMLWidget(value=_header_html(m["label"], m["model"])) for m in MODELS]
panel_blocks = [
    VBox([headers[i], panels[i]], layout=Layout(border="1px solid #eee", padding="4px", border_radius="4px"))
    for i in range(len(MODELS))
]
grid = GridBox(
    panel_blocks,
    layout=Layout(grid_template_columns="repeat(2, 1fr)", grid_gap="8px"),
)

vote_btns = [Button(description=f"Vote {m['label']}") for m in MODELS]
judge_btn = Button(description="🧑‍⚖️ LLM judge")
vote_bar = HBox([*vote_btns, judge_btn])

scoreboard = Output()
judge_out = Output()

image_panel = VBox([
    HTMLWidget(value='<b style="font-family:sans-serif;font-size:13px;">Image</b>'),
    image_preview,
    image_upload,
    HBox([HTMLWidget(value='<span style="font-family:sans-serif;font-size:12px;color:#666;">or URL:</span>'), image_url_box]),
])

display(VBox([
    image_panel,
    prompt_box,
    HBox([fight_btn, clear_btn]),
    grid,
    vote_bar,
    scoreboard,
    judge_out,
]))

## 4. Wire it up

Fight → fan out to all vision models in parallel via `asyncio.gather`, each receiving the same image + prompt.
Vote increments a model's score; LLM judge picks one model and explains why.

In [ ]:
import asyncio

import nest_asyncio
from IPython.display import HTML, clear_output, display

nest_asyncio.apply()

scores = {m["label"]: 0 for m in MODELS}
last_round = {}


def render_scoreboard():
    max_score = max(scores.values()) if scores.values() else 0
    rows = ""
    for label, n in sorted(scores.items(), key=lambda kv: -kv[1]):
        width = int(n / max_score * 200) if max_score else 0
        rows += (
            '<div style="margin:4px 0;display:flex;align-items:center;gap:8px;'
            'font-family:sans-serif;font-size:13px;">'
            f'  <div style="width:80px;">{label}</div>'
            f'  <div style="height:18px;width:{width}px;background:#007bff;'
            'border-radius:4px;transition:width 0.3s;"></div>'
            f'  <div style="margin-left:6px;font-weight:600;">{n}</div>'
            '</div>'
        )
    with scoreboard:
        clear_output()
        display(HTML(
            f'<div><b style="font-family:sans-serif;font-size:13px;">Scoreboard</b>{rows}</div>'
        ))


def _highlight_winner(label):
    for i, m in enumerate(MODELS):
        if m["label"].lower() == label.lower():
            panel_blocks[i].layout.border = "2px solid #28a745"
        else:
            panel_blocks[i].layout.border = "1px solid #eee"


def _reset_borders():
    for pb in panel_blocks:
        pb.layout.border = "1px solid #eee"


async def run_round(prompt_text, image_url):
    _reset_borders()
    async with aiohttp.ClientSession() as session:
        tasks = [
            stream_model(session, MODELS[i], prompt_text, image_url, panels[i], headers[i])
            for i in range(len(MODELS))
        ]
        return await asyncio.gather(*tasks)


def on_fight(_):
    prompt_text = prompt_box.value.strip()
    if not prompt_text:
        return
    try:
        image_url = _resolve_image_url()
    except Exception as e:
        with judge_out:
            clear_output()
            display(HTML(
                f'<div style="color:#dc3545;font-family:monospace;font-size:12px;">'
                f'Could not load image: {e}</div>'
            ))
        return
    if not image_url:
        return
    judge_out.clear_output()
    results = asyncio.run(run_round(prompt_text, image_url))
    last_round.clear()
    last_round.update({r["label"]: r["text"] for r in results})


def on_clear(_):
    for i, m in enumerate(MODELS):
        panels[i].clear_output()
        headers[i].value = _header_html(m["label"], m["model"])
    _reset_borders()
    judge_out.clear_output()
    last_round.clear()


def make_vote_handler(label):
    def handler(_):
        scores[label] += 1
        _highlight_winner(label)
        render_scoreboard()
    return handler


async def call_judge(prompt_text, responses):
    blocks = "\n\n".join(f"### {label}\n{text}" for label, text in responses.items())
    judge_prompt = (
        f"User asked about an image:\n{prompt_text}\n\n"
        f"Candidate answers from vision models:\n{blocks}\n\n"
        "Which answer is most accurate and useful? "
        "First write one sentence of reasoning. "
        "Then on a new line, output ONLY the label of the winner (one word)."
    )
    async with aiohttp.ClientSession() as session:
        headers_dict = {
            "Authorization": f"Bearer {EDENAI_API_KEY}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": JUDGE_MODEL,
            "messages": [{"role": "user", "content": judge_prompt}],
        }
        async with session.post(EDENAI_URL, headers=headers_dict, json=payload) as r:
            data = await r.json()
            return data["choices"][0]["message"]["content"].strip()


def on_judge(_):
    if not last_round:
        return
    response = asyncio.run(call_judge(prompt_box.value.strip(), last_round))
    lines = [ln.strip() for ln in response.splitlines() if ln.strip()]
    winner_line = lines[-1] if lines else response
    reasoning = " ".join(lines[:-1]) if len(lines) > 1 else ""

    chosen = None
    for label in scores:
        if label.lower() in winner_line.lower():
            scores[label] += 1
            chosen = label
            break

    with judge_out:
        clear_output()
        reasoning_html = (
            f'<div style="font-family:sans-serif;font-size:12px;color:#555;margin-top:4px;">'
            f'{reasoning}</div>'
        ) if reasoning else ""
        winner_html = (
            f'<div style="font-family:sans-serif;font-size:13px;">'
            f'<b>🧑‍⚖️ Judge picked:</b> {chosen or winner_line}</div>'
        )
        display(HTML(
            f'<div style="background:#f8f9fa;padding:10px 12px;border-radius:4px;'
            f'border-left:4px solid #28a745;">{winner_html}{reasoning_html}</div>'
        ))

    if chosen:
        _highlight_winner(chosen)
        render_scoreboard()


fight_btn.on_click(on_fight)
clear_btn.on_click(on_clear)
judge_btn.on_click(on_judge)
for btn, m in zip(vote_btns, MODELS):
    btn.on_click(make_vote_handler(m["label"]))

## 5. Try these prompts

Different prompt types stress different parts of a vision model. The same image can produce wildly different answers depending on what you ask.

- **Describe:** *Describe this image in one sentence.*
- **Count:** *How many distinct objects are visible? List them.*
- **OCR:** *Transcribe every piece of text you see, line by line.* (use a photo of a sign, menu, or receipt)
- **Structured:** *Return a JSON object with keys: dominant_colors (list of 3 hex), main_subject (string), setting ("indoor"/"outdoor"/"unknown").*
- **Spatial reasoning:** *Is there anything in the upper-left quadrant? What about the lower-right?*
- **Style transfer prompt:** *Describe this image as a haiku.*

## 6. Customize

- **Different lineup:** swap any model in `MODELS` — every entry only needs to have `image` in `input_modalities`. The full vision catalog is at `GET /v3/llm/models` (filter for `capabilities.input_modalities` containing `"image"`).
- **More than 4 models:** add rows — the grid auto-resizes.
- **Persist scores across sessions:** dump `scores` to JSON between fights to keep a leaderboard.
- **Batch evaluate on a folder of images:** loop `run_round` over a directory and write results to CSV — turns this notebook into a tiny eval harness.

---

Tried it? [Open an issue](https://github.com/) with the image + prompt that surprised you most.